# Phase 2 — arch 2 (LLaMA-style) FULL fine-tuning, bidirectional (clean run)

Run All. Inputs: `datasetss_final` (arch2e2_ckpt.pt, arch3_mlm_best.pt, legal_bpe_merges.json), `HF_TOKEN` secret, Internet on, **GPU**. Same recipe as my arch-3 full-FT submission; only the backbone differs.

In [ ]:
import json, os, glob, collections
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "Exploration-Lab/CS-781-Capstone"
files = ["Train/fold1-text.json", "Train/fold1-gold.json",
         "Train/fold2-text.json", "Train/fold2-gold.json",
         "Train/labels.txt", "Test/fold3-text.json"]
P = {}
for f in files:
    P[f] = hf_hub_download(REPO, f"Phase_2_LNER/{f}", repo_type="dataset", token=HF_TOKEN)
    print(f, "->", os.path.getsize(P[f]) // 1024, "KB")

In [ ]:
t1 = json.load(open(P["Train/fold1-text.json"]))
g1 = json.load(open(P["Train/fold1-gold.json"]))
labels = open(P["Train/labels.txt"]).read().split()

print("labels:", labels)
print("text type:", type(t1).__name__, "| gold type:", type(g1).__name__)
k = list(t1)[0] if isinstance(t1, dict) else 0
print("first id:", k)
print("text sample:", repr(str(t1[k])[:400]))
print("gold sample:", json.dumps(g1[k] if isinstance(g1, dict) else g1[0], ensure_ascii=False)[:800])

In [ ]:
def load(fold):
    t = json.load(open(P[f"Train/{fold}-text.json"]))
    g = json.load(open(P[f"Train/{fold}-gold.json"]))
    return t, g

for fold in ("fold1", "fold2"):
    t, g = load(fold)
    lens = [len(v) for v in t.values()]
    print(f"{fold}: {len(t)} docs | chars per doc: min {min(lens)}, median {sorted(lens)[len(lens)//2]}, max {max(lens)}")

t3 = json.load(open(P["Test/fold3-text.json"]))
lens = [len(v) for v in t3.values()]
print(f"fold3 (test): {len(t3)} docs | chars: min {min(lens)}, median {sorted(lens)[len(lens)//2]}, max {max(lens)}")

In [ ]:
def stats(fold):
    t, g = load(fold)
    cnt, lens, bad, edge_ws, overlaps = collections.Counter(), collections.defaultdict(list), 0, 0, 0
    for d, spans in g.items():
        text = t[d]
        spans = sorted(spans, key=lambda s: (s[0], s[1]))
        for i, (s, e, lab, ent) in enumerate(spans):
            cnt[lab] += 1
            lens[lab].append(e - s)
            if text[s:e] != ent: bad += 1                       # is `end` exclusive?
            if text[s:e] != text[s:e].strip(): edge_ws += 1      # spaces at span edges?
            if i and s < spans[i-1][1]: overlaps += 1            # overlapping / nested spans?
    print(f"\n{fold}: {sum(cnt.values())} entities | text mismatches {bad} | edge-whitespace {edge_ws} | overlaps {overlaps}")
    for lab in labels:
        L = lens[lab]
        print(f"  {lab:10s} {cnt[lab]:5d}   avg len {sum(L)/max(1,len(L)):6.1f}   max len {max(L) if L else 0}")
    return cnt

c1, c2 = stats("fold1"), stats("fold2")
print("\nempty test docs:", [d for d, v in t3.items() if not v.strip()])

In [ ]:
import re
ROLE_LABELS = ("APP", "RESP", "WIT")

def name_consistency(fold):
    t, g = load(fold); tot = agree = 0; conflicts = collections.Counter()
    for d, spans in g.items():
        by = collections.defaultdict(collections.Counter)
        for s, e, l, ent in spans:
            if l in ROLE_LABELS + ("A.COUNSEL", "R.COUNSEL", "JUDGE"): by[ent][l] += 1
        for ent, c in by.items():
            n = sum(c.values()); tot += n; agree += c.most_common(1)[0][1]
            if len(c) > 1: conflicts[tuple(sorted(c))] += 1
    print(f"{fold}: person mentions {tot} | same role as the name's majority: {agree/tot:.3f} | conflicts {conflicts.most_common(4)}")

def unlabelled_repeats(fold, min_len=4):
    t, g = load(fold); lab = unl = 0
    for d, spans in g.items():
        text = t[d]; gold = {(s, e) for s, e, *_ in spans}
        for ent in {ent for s, e, l, ent in spans if l in ROLE_LABELS and len(ent) >= min_len}:
            for m in re.finditer(r"(?<!\w)" + re.escape(ent) + r"(?!\w)", text):
                if (m.start(), m.end()) in gold: lab += 1
                else: unl += 1
    print(f"{fold}: repeats of APP/RESP/WIT names → labelled {lab} | NOT labelled {unl} | precision ceiling {lab/(lab+unl):.3f}")

for f in ("fold1", "fold2"):
    name_consistency(f)
    unlabelled_repeats(f)

In [ ]:
%%writefile a1_tokenizer.py
"""Expose your Assignment-1 BPE implementation to Assignment 3.

Reuse your A-1 tokenizer code/class here. The important change is that it must be
RETRAINED on the `train_text` supplied by Assignment 3 rather than reusing a tokenizer
trained on the Assignment-1 dataset.
"""
import re
from collections import Counter

PAT=re.compile(r" ?[A-Za-z]+| ?\d+| ?[^\sA-Za-z\d]+|\s+")
class BPETokeniser:#defining the class
    def __init__(self):
        self.merges={}
        self.vocab={i: bytes([i]) for i in range(256)}
        self.cache={}

    @property
    def vocab_size(self):#vocab size
        return len(self.vocab)

    def _count_pairs(self,words,counts):#how actuall work we start counting how many words have same pair 
        pair_counts={}
        for word,count in zip(words,counts):
            for a,b in zip(word,word[1:]):
                pair_counts[(a,b)]=pair_counts.get((a,b),0)+count
        return pair_counts
    def _merge(self,ids,pair,new_id):
        out=[]
        i=0
        while i<len(ids):
            if i+1<len(ids) and ids[i]==pair[0] and ids[i+1]==pair[1]:
                out.append(new_id)
                i+=2
            else:
                out.append(ids[i])
                i+=1
        return out
    def train(self,text,vocab_size):
        chunks=PAT.findall(text)
        chunk_counts=Counter(chunks)
        words=[]
        counts=[]
        for chunk,n in chunk_counts.items():
            words.append(list(chunk.encode("utf-8")))
            counts.append(n)
        for r in range(vocab_size-256):
                pair_counts=self._count_pairs(words,counts)
                if not pair_counts:
                    break
                best = min(pair_counts, key=lambda p: (-pair_counts[p], p))
                new_id=256+r
                self.merges[best]=new_id
                self.vocab[new_id]=self.vocab[best[0]] + self.vocab[best[1]]
                words=[self._merge(w,best,new_id) for w in words]
    def _encode_chunk(self,chunk):
        if chunk in self.cache:
            return self.cache[chunk]
        ids=list(chunk.encode("utf-8"))
        while True:
            candidates=[p for p in zip(ids,ids[1:]) if p in self.merges]
            if not candidates:
                break
            best=min(candidates,key=lambda p:self.merges[p])
            ids=self._merge(ids,best,self.merges[best])
        self.cache[chunk]=ids
        return ids
    def encode(self,text):
        ids=[]
        for chunk in PAT.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids
    def decode(self,ids):
        data=b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8",errors="replace")
             
         

def train_tokenizer(train_text: str, vocab_size: int):
    """Train your A-1 BPE tokenizer on `train_text` and return the trained tokenizer.

    Required returned interface:
      tokenizer.encode(text) -> sequence of integer ids
      tokenizer.decode(ids) -> string
      tokenizer.vocab_size OR tokenizer.get_vocab_size()

    Replace this stub with your Assignment-1 tokenizer implementation/training call.
    """
    tok=BPETokeniser()
    tok.train(train_text,vocab_size)
    return tok

In [ ]:
import re
from a1_tokenizer import BPETokeniser
mp = glob.glob("/kaggle/input/**/legal_bpe_merges.json", recursive=True)[0]
raw = json.load(open(mp))
print(type(raw).__name__, len(raw))

In [ ]:
def load_tokenizer(raw):
    tok = BPETokeniser()
    if isinstance(raw, dict) and "merges" in raw:
        raw = raw["merges"]
    triples = []
    if isinstance(raw, dict):                                  # {"a,b": new_id}
        for k, v in raw.items():
            a, b = map(int, re.split(r"[,\s]+", k.strip("()[] ")))
            triples.append((a, b, int(v)))
    else:                                                      # list form
        for i, m in enumerate(raw):
            triples.append((int(m[0]), int(m[1]), int(m[2]) if len(m) == 3 else 256 + i))
    for a, b, new in sorted(triples, key=lambda t: t[2]):      # rebuild in learned order
        tok.merges[(a, b)] = new
        tok.vocab[new] = tok.vocab[a] + tok.vocab[b]
    return tok

tok = load_tokenizer(raw)
print("merges:", len(tok.merges), "| vocab:", tok.vocab_size)   # expect 65,098 | 65,354

In [ ]:
from a1_tokenizer import PAT
import bisect

def encode_with_offsets(text):
    ids, offs = [], []
    for m in PAT.finditer(text):
        chunk, base = m.group(), m.start()
        b2c = []                                         # byte index -> char index within chunk
        for ci, ch in enumerate(chunk):
            b2c += [ci] * len(ch.encode("utf-8"))
        b2c.append(len(chunk))
        b = 0
        for t in tok.encode(chunk):
            nb = len(tok.vocab[t])
            ids.append(t)
            offs.append((base + b2c[b], base + b2c[min(b + nb, len(b2c) - 1)]))
            b += nb
    return ids, offs

# check: chunk-wise encoding must equal encoding the whole text
t1, g1 = load("fold1")
d0 = list(t1)[0]
ids, offs = encode_with_offsets(t1[d0][:5000])
print("same as tok.encode:", ids == tok.encode(t1[d0][:5000]), "|", len(ids), "tokens")
print("offsets cover the text:", offs[0][0] == 0, all(offs[i][1] == offs[i+1][0] for i in range(len(offs)-1)))
print([(t1[d0][s:e]) for s, e in offs[:15]])

In [ ]:
TAGS = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
T2I = {t: i for i, t in enumerate(TAGS)}

def core_start(text, s, e):                    # first non-space char of a token
    seg = text[s:e]
    return s + (len(seg) - len(seg.lstrip()))

def tag_doc(text, offs, spans):
    tags = [0] * len(offs)
    starts = [core_start(text, s, e) for s, e in offs]
    ends = [e for _, e in offs]
    for s, e, lab, *_ in spans:
        i = bisect.bisect_right(ends, s)       # first token ending after entity start
        first = True
        while i < len(offs) and starts[i] < e:
            if starts[i] < ends[i]:            # skip pure-whitespace tokens
                tags[i] = T2I[("B-" if first else "I-") + lab]; first = False
            i += 1
    return tags

def decode(text, offs, tags):
    spans, cur = [], None
    for i, t in enumerate(tags):
        name = TAGS[t]
        if name == "O":
            if cur: spans.append(cur); cur = None
            continue
        p, lab = name.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [core_start(text, *offs[i]), offs[i][1], lab]
        else:
            cur[1] = offs[i][1]
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:                    # trim whitespace at the edges
        while e > s and text[e-1].isspace(): e -= 1
        if e > s: out.append([s, e, lab])
    return out

def strict_macro_f1(gold, pred):
    tp, fp, fn = collections.Counter(), collections.Counter(), collections.Counter()
    for d in gold:
        G = {(s, e, l) for s, e, l, *_ in gold[d]}
        Pd = {(s, e, l) for s, e, l, *_ in pred.get(d, [])}
        for x in Pd: (tp if x in G else fp)[x[2]] += 1
        for x in G - Pd: fn[x[2]] += 1
    per = {}
    for l in labels:
        p = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        r = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = 2 * p * r / (p + r) if p + r else 0.0
    return sum(per.values()) / len(labels), per

In [ ]:
import time
ENC = {}
for fold in ("fold1", "fold2"):
    t, g = load(fold)
    t0 = time.time()
    pred = {}
    for d, text in t.items():
        ids, offs = encode_with_offsets(text)
        ENC[d] = (ids, offs)
        pred[d] = decode(text, offs, tag_doc(text, offs, g.get(d, [])))
    f, per = strict_macro_f1(g, pred)
    print(f"{fold}: oracle strict macro-F1 = {f:.4f}  ({time.time()-t0:.0f}s)")
    print("   ", {l: round(v, 3) for l, v in per.items()})

In [ ]:
def boundary_misses(fold, max_show=12):
    t, g = load(fold)
    miss = collections.Counter(); examples = collections.defaultdict(list)
    for d, spans in g.items():
        text = t[d]; ids, offs = ENC[d]
        starts = {core_start(text, s, e) for s, e in offs}
        ends = {e for _, e in offs}
        for s, e, lab, ent in spans:
            ok_s, ok_e = s in starts, e in ends
            if ok_s and ok_e: continue
            kind = ("START" if not ok_s else "") + ("END" if not ok_e else "")
            miss[(lab, kind)] += 1
            if len(examples[lab]) < max_show:
                i = next(k for k, (a, b) in enumerate(offs) if b > (s if not ok_s else e - 1))
                examples[lab].append(f"{kind:8s} gold={text[s:e]!r}  context={text[max(0,s-15):e+15]!r}  "
                                     f"token={text[offs[i][0]:offs[i][1]]!r}")
    print(fold, "misaligned:", dict(miss))
    for lab, ex in examples.items():
        print(f"\n--- {lab}"); print("\n".join(ex))

boundary_misses("fold1")

In [ ]:
def build_known(folds):
    known = set()
    for fold in folds:
        _, g = load(fold)
        for spans in g.values():
            for s, e, lab, ent in spans:
                known.add((lab, ent))
    return known

def refine(text, s, e, lab, known):
    # 1) trailing ,;:([ glued after the entity
    while e > s and text[e-1] in ",;:([":
        e -= 1
    # 2) a '.' glued after a closing bracket:  ']. ' ')." 
    while e - s >= 2 and text[e-1] == "." and text[e-2] in ")]":
        e -= 1
    # 3) plural / adjective suffix, only if the shorter form is a known entity and the longer is not
    w = text[s:e]
    for suf in ("s", "al"):
        if w.endswith(suf) and (lab, w) not in known and (lab, w[:-len(suf)]) in known:
            e -= len(suf)
            break
    return s, e

def decode_refined(text, offs, tags, known):
    out = []
    for s, e, lab in decode(text, offs, tags):
        s, e = refine(text, s, e, lab, known)
        if e > s:
            out.append([s, e, lab])
    return out

KNOWN_ALL = build_known(["fold1", "fold2"])     # oracle only; in real CV build from the TRAINING fold only
for fold in ("fold1", "fold2"):
    t, g = load(fold)
    pred = {d: decode_refined(text, ENC[d][1], tag_doc(text, ENC[d][1], g.get(d, [])), KNOWN_ALL)
            for d, text in t.items()}
    f, per = strict_macro_f1(g, pred)
    print(f"{fold}: refined oracle = {f:.4f}  | PREC {per['PREC']:.3f}  AUTH {per['AUTH']:.3f}  "
          f"COURT {per['COURT']:.3f}  STAT {per['STAT']:.3f}")

In [ ]:
import math, time, random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict
device = "cuda"
EOS_ID = 65354

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

In [ ]:
def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

In [ ]:
def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

In [ ]:
def prefix_mask(T,prefix_lens,device=None):
    causal=torch.tril(torch.ones(T,T,dtype=torch.bool,device=device))
    pos=torch.arange(T,device=device)
    in_prefix=pos[None,:]<prefix_lens[:,None]
    both=in_prefix[:,:,None] & in_prefix[:,None,:]
    return (causal[None] | both)[:, None]
m = prefix_mask(6, torch.tensor([3, 0, 6]))
print(tuple(m.shape)); print(m[0, 0].int())
print(torch.equal(m[1, 0], torch.tril(torch.ones(6, 6, dtype=torch.bool))), bool(m[2, 0].all()), not bool(m[0, 0, 3, 4]))

In [ ]:
class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x,mask=None):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        if mask is None:
            mask=self.causal_mask[:T,:T]
        scores=scores.masked_fill(~mask,float("-inf"))
        
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

In [ ]:
from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0

class LlamaBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x,mask=None):
        x=x+self.attn(self.norm1(x),mask)
        x=x+self.ffn(self.norm2(x))
        return x

class PrefixLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx,prefix_lens=None):
        if prefix_lens is not None:
            mask=prefix_mask(idx.size(1),prefix_lens,idx.device)
        else:
            mask=None
        x=self.tok_emb(idx)
        for block in self.blocks: x=block(x,mask)
        x=self.norm_f(x)
        return self.head(x)

In [ ]:
def load_encoder(path):
    ck = torch.load(path, map_location="cpu")
    cfg = LlamaConfig(**ck["cfg"])
    enc = PrefixLM(cfg); enc.load_state_dict(ck["model"])
    print(path.split("/")[-1], "| step", ck.get("step"), "| best_val", round(ck.get("best_val", float("nan")), 3),
          "| vocab", cfg.vocab_size, f"| {sum(p.numel() for p in enc.parameters()):,} params")
    del ck
    return enc, cfg

CK_A2 = glob.glob("/kaggle/input/**/arch2e2_ckpt.pt", recursive=True)[0]
CK_A3 = glob.glob("/kaggle/input/**/arch3_mlm_best.pt", recursive=True)[0]
for p in (CK_A2, CK_A3):
    e, c = load_encoder(p); del e

In [ ]:
W, STRIDE = 1024,512        # your choice (e.g. 512)

def make_windows(ids, tags=None):
    n, out, s = len(ids), [], 0
    while True:
        e = min(s + W, n)
        out.append((s, ids[s:e], None if tags is None else tags[s:e]))
        if e == n: break
        s += STRIDE
    return out

def build_split(folds):
    data = []
    for fold in folds:
        t, g = load(fold)
        for d, text in t.items():
            ids, offs = ENC[d]
            for s, wi, wt in make_windows(ids, tag_doc(text, offs, g.get(d, []))):
                data.append((d, s, wi, wt))
    return data

In [ ]:
w1, w2 = build_split(["fold1"]), build_split(["fold2"])
print(f"windows: fold1 {len(w1)} | fold2 {len(w2)}")
print("tokens: fold1", sum(len(ENC[d][0]) for d in load('fold1')[0]),
      "| fold2", sum(len(ENC[d][0]) for d in load('fold2')[0]))
t, g = load("fold1"); ok = True
for d, text in t.items():
    ids, offs = ENC[d]; full = tag_doc(text, offs, g.get(d, []))
    cover = [False] * len(ids)
    for s, wi, wt in make_windows(ids, full):
        for k in range(len(wi)): cover[s + k] = True
        ok &= (wi == ids[s:s+len(wi)]) and (wt == full[s:s+len(wt)])
    ok &= all(cover)
print("full coverage + tags consistent:", ok)
cnt = collections.Counter(tg for *_, wt in w1 + w2 for tg in wt)
print({TAGS[k]: v for k, v in sorted(cnt.items())})

In [ ]:
class Tagger(nn.Module):
    def __init__(self,encoder,d_model,n_tags,dropout):
        super().__init__()
        self.encoder=encoder
        self.drop=nn.Dropout(dropout)
        self.cls=nn.Linear(d_model,n_tags)

    def forward(self,idx,prefix_lens=None):
        e=self.encoder
        mask=prefix_mask(idx.size(1),prefix_lens,idx.device) if prefix_lens is not None else None
        x=e.tok_emb(idx)
        for block in e.blocks:
            x = block(x, mask)
        x=e.norm_f(x)
        return self.cls(self.drop(x))

In [ ]:
def collate(batch):
    L = max(len(b[2]) for b in batch)
    x = torch.full((len(batch), L), EOS_ID, dtype=torch.long)
    y = torch.full((len(batch), L), -100, dtype=torch.long)
    pl = torch.tensor([len(b[2]) for b in batch])
    for i, (_, _, ids, tg) in enumerate(batch):
        x[i, :len(ids)] = torch.tensor(ids)
        if tg is not None: y[i, :len(tg)] = torch.tensor(tg)
    return x.to(device), y.to(device), pl.to(device)

def class_weights(windows, power):
    cnt = torch.zeros(len(TAGS))
    for *_, tg in windows:
        cnt += torch.bincount(torch.tensor(tg), minlength=len(TAGS)).float()
    w = (cnt.sum() / cnt.clamp(min=1)) ** power
    return (w / w.mean()).to(device)

@torch.no_grad()
def predict_logits(model, ids, bidir, B=8):
    wins = make_windows(ids)
    n = len(ids); out = torch.zeros(n, len(TAGS)); best = torch.full((n,), float("inf"))
    model.eval()
    for i in range(0, len(wins), B):
        chunk = wins[i:i+B]
        x, _, pl = collate([(None, s, w, None) for s, w, _ in chunk])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(x, pl if bidir else None).float().cpu()
        for j, (s, w, _) in enumerate(chunk):
            L = len(w); pos = torch.arange(L)
            dist = (pos - (L - 1) / 2).abs()
            idx = s + pos; better = dist < best[idx]
            out[idx[better]] = lg[j, :L][better]; best[idx[better]] = dist[better]
    model.train()
    return out

def predict_texts(model, texts, bidir, known):
    pred = {}
    for d, text in texts.items():
        if not text.strip(): pred[d] = []; continue
        if d not in ENC: ENC[d] = encode_with_offsets(text)
        ids, offs = ENC[d]
        tags = predict_logits(model, ids, bidir).argmax(-1).tolist()
        pred[d] = decode_refined(text, offs, tags, known)
    return pred

def train_tagger(model, train_w, dev_fold, bidir, known, *, EPOCHS, LR_ENC, LR_HEAD,
                 WARMUP_FRAC, MICRO_B, CLASS_W_POWER, WD, CLIP=1.0, seed=0):
    torch.manual_seed(seed); random.seed(seed)
    train_w = list(train_w)
    cw = class_weights(train_w, CLASS_W_POWER)
    head = list(model.cls.parameters()); hid = {id(p) for p in head}
    body = [p for p in model.parameters() if id(p) not in hid and p.requires_grad]
    opt = torch.optim.AdamW([{"params": body, "lr": LR_ENC}, {"params": head, "lr": LR_HEAD}], weight_decay=WD)
    per_ep = math.ceil(len(train_w) / MICRO_B); total = EPOCHS * per_ep; warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler("cuda")
    dev_t, dev_g = load(dev_fold)
    best_f, best_state, best_per = -1.0, None, None
    model.train()
    for ep in range(EPOCHS):
        random.shuffle(train_w); t0 = time.time(); tot = 0.0
        for i in range(0, len(train_w), MICRO_B):
            x, y, pl = collate(train_w[i:i + MICRO_B])
            with torch.autocast("cuda", dtype=torch.float16):
                lg = model(x, pl if bidir else None)
            loss = F.cross_entropy(lg.float().view(-1, len(TAGS)), y.view(-1), weight=cw, ignore_index=-100)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            tot += loss.item()
        f, per = strict_macro_f1(dev_g, predict_texts(model, dev_t, bidir, known))
        print(f"epoch {ep+1}/{EPOCHS} | train loss {tot/per_ep:.4f} | dev strict macro-F1 {f:.4f} | {time.time()-t0:.0f}s")
        if f > best_f:
            best_f, best_per = f, per
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print("BEST dev F1", round(best_f, 4), {l: round(v, 3) for l, v in best_per.items()})
    return best_f

In [ ]:
# ---------- constrained (Viterbi) decoding: valid BIO + whole-word consistency ----------
NT = len(TAGS)
ent_of = [None] + [TAGS[i][2:] for i in range(1, NT)]
NEG = -1e9
allow_start = np.ones((NT, NT), bool)          # token that STARTS a word
allow_cont  = np.zeros((NT, NT), bool)         # token that CONTINUES a word (e.g. "annum" after "Tar")
for p in range(NT):
    for c in range(NT):
        if TAGS[c].startswith("I-"):
            allow_start[p, c] = (p != 0 and ent_of[p] == ent_of[c])
        allow_cont[p, c] = (c == 0) if p == 0 else (TAGS[c] == "I-" + ent_of[p])
T_START = np.where(allow_start, 0.0, NEG); T_CONT = np.where(allow_cont, 0.0, NEG)
FIRST_OK = np.array([not t.startswith("I-") for t in TAGS])

def viterbi(logp, cont):
    n = logp.shape[0]
    score = np.where(FIRST_OK, logp[0], NEG)
    back = np.zeros((n, NT), np.int16)
    for i in range(1, n):
        cand = score[:, None] + (T_CONT if cont[i] else T_START)
        back[i] = cand.argmax(0)
        score = cand.max(0) + logp[i]
    tags = [int(score.argmax())]
    for i in range(n - 1, 0, -1):
        tags.append(int(back[i, tags[-1]]))
    return tags[::-1]

def fix_brackets(text, spans):                 # drop an unmatched closing ] or ) at the end
    out = []
    for s, e, l in spans:
        while e > s and text[e-1] in ")]" and text[s:e].count(text[e-1]) > text[s:e].count("(" if text[e-1] == ")" else "["):
            e -= 1
        if e > s: out.append([s, e, l])
    return out

def predict_texts_v(model, texts, bidir, known):
    pred = {}
    for d, text in texts.items():
        if not text.strip(): pred[d] = []; continue
        if d not in ENC: ENC[d] = encode_with_offsets(text)
        ids, offs = ENC[d]
        logp = torch.log_softmax(predict_logits(model, ids, bidir), -1).numpy()
        cont = [False] + [offs[i-1][1] == offs[i][0] and text[offs[i][0]-1].isalnum() and text[offs[i][0]].isalnum()
                          for i in range(1, len(offs))]
        pred[d] = fix_brackets(text, decode_refined(text, offs, viterbi(logp, cont), known))
    return pred


In [ ]:
ROLE_SET = ("APP", "RESP", "A.COUNSEL", "R.COUNSEL")
A_W = r"(appellant|petitioner|applicant|plaintiff|complainant)"
R_W = r"(respondent|opposite part|defendant)"
LAB = {"C": {"A": "A.COUNSEL", "R": "R.COUNSEL"}, "P": {"A": "APP", "R": "RESP"}}

def side_cues(text, s, e):
    cues = []
    after = text[e:e+150].lower(); tail = text[max(0, s-40):s].lower()
    m_a = re.search(r"\b(for|behalf of)\b[^;]{0,60}?" + A_W, after)     # counsel: "...appearing for the appellant"
    m_r = re.search(r"\b(for|behalf of)\b[^;]{0,60}?" + R_W, after)
    if m_a and (not m_r or m_a.start() < m_r.start()): cues.append(("C", "A", 3))
    elif m_r: cues.append(("C", "R", 3))
    if re.search(A_W + r"[^;\n]{0,25}$", tail) or "instance of" in tail: cues.append(("P", "A", 3))   # "appellant No.4 - X"
    elif re.search(R_W + r"[^;\n]{0,25}$", tail): cues.append(("P", "R", 3))
    if s < 2500:                                                        # header "X Vs. Y"
        if re.match(r"\W{0,5}(&\s*ors\.?)?\W{0,3}(vs\.?|v\.|versus)\b", text[e:e+20].lower()): cues.append(("P", "A", 5))
        if re.search(r"\b(vs\.?|v\.|versus)\W{0,5}$", text[max(0, s-15):s].lower()): cues.append(("P", "R", 5))
    return cues

def apply_roles(text, spans):
    votes = collections.defaultdict(collections.Counter)
    for s, e, l in spans:
        if l not in ROLE_SET: continue
        k = text[s:e]; votes[k][l] += 1
        for g, side, w in side_cues(text, s, e):
            votes[k][LAB[g][side]] += w
    role = {k: c.most_common(1)[0][0] for k, c in votes.items()}
    sur = collections.defaultdict(set)
    for k, l in role.items():
        w = k.split()
        if len(w) >= 2 and len(w[-1]) >= 4: sur[w[-1]].add(l)
    sur = {k: next(iter(v)) for k, v in sur.items() if len(v) == 1}
    out = []
    for s, e, l in spans:
        if l in ROLE_SET:
            k = text[s:e]
            l = role.get(k, l)
            if len(k.split()) == 1 and k in sur: l = sur[k]
        out.append([s, e, l])
    return out


In [ ]:
# ===== helpers needed by the PEFT runs (plain-window prediction + ensemble decoding) =====
@torch.no_grad()
def predict_logits_plain(model, ids, bidir, B=8):
    wins = make_windows(ids)
    n = len(ids); out = torch.zeros(n, len(TAGS)); best = torch.full((n,), float("inf"))
    model.eval()
    for i in range(0, len(wins), B):
        chunk = wins[i:i + B]
        x, _, pl = collate([(None, s, w, None) for s, w, _ in chunk])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(x, pl if bidir else None).float().cpu()
        for j, (s, w, _) in enumerate(chunk):
            L = len(w); pos = torch.arange(L); dist = (pos - (L - 1) / 2).abs()
            idx = s + pos; better = dist < best[idx]
            out[idx[better]] = lg[j, :L][better]; best[idx[better]] = dist[better]
    return out

def ens_predict_b(members, texts, known):
    pred = {}
    for d, text in texts.items():
        if not text.strip(): pred[d] = []; continue
        if d not in ENC: ENC[d] = encode_with_offsets(text)
        ids, offs = ENC[d]
        logp = sum(torch.log_softmax(fn(m, ids, b), -1) for m, fn, b in members) / len(members)
        cont = [False] + [offs[i-1][1] == offs[i][0] and text[offs[i][0]-1].isalnum() and text[offs[i][0]].isalnum()
                          for i in range(1, len(offs))]
        pred[d] = fix_brackets(text, decode_refined(text, offs, viterbi(logp.numpy(), cont), known))
    return pred
dev_t, dev_g = load("fold2")
print("helpers ready")


In [ ]:
# ===== arch 2 FULL fine-tuning, bidirectional: train on both folds, predict test, export probabilities =====
import zipfile, numpy as np
TRAIN_FOLDS = ["fold1", "fold2"]
torch.cuda.empty_cache()
enc2, ecfg2 = load_encoder(CK_A2)
m_full = Tagger(enc2, ecfg2.d_model, len(TAGS), dropout=0.1).to(device)
w_train = [w for f in TRAIN_FOLDS for w in build_split([f])]
train_tagger(m_full, w_train, "fold2", bidir=True, known=build_known(TRAIN_FOLDS),
             EPOCHS=10, LR_ENC=1e-5, LR_HEAD=1e-4, WARMUP_FRAC=0.1, MICRO_B=4,
             CLASS_W_POWER=0.2, WD=0.01)               # same settings as my arch-3 full-FT submission
torch.save(m_full.state_dict(), "/kaggle/working/tagger_arch2_full_both.pt")
m_full.eval()

t3 = json.load(open(P["Test/fold3-text.json"]))
known = build_known(TRAIN_FOLDS)
pred, LOGP = {}, {}
for d, text in t3.items():
    if not text.strip(): pred[d] = []; continue
    if d not in ENC: ENC[d] = encode_with_offsets(text)
    ids, offs = ENC[d]
    logp = torch.log_softmax(predict_logits_plain(m_full, ids, True), -1)
    LOGP[d] = logp.numpy().astype(np.float16)                       # for later cross-model ensembling
    cont = [False] + [offs[i-1][1] == offs[i][0] and text[offs[i][0]-1].isalnum() and text[offs[i][0]].isalnum()
                      for i in range(1, len(offs))]
    pred[d] = fix_brackets(text, decode_refined(text, offs, viterbi(logp.numpy(), cont), known))
pred = {d: apply_roles(t3[d], p) for d, p in pred.items()}
np.savez_compressed("/kaggle/working/logp_test_arch2_full.npz", **LOGP)
json.dump(pred, open("/kaggle/working/prediction_arch2_full.json", "w"))
with zipfile.ZipFile("/kaggle/working/submission_arch2_full.zip", "w") as z:
    z.write("/kaggle/working/prediction_arch2_full.json", arcname="prediction.json")
print("ARCH2 FULL-FT | docs:", len(pred), "| entities:", sum(len(v) for v in pred.values()), "| empty doc:", pred.get("1305957"))
